<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Political_Violence_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba por Ciudad

In [ ]:
import requests
from bs4 import BeautifulSoup

def violence_news_index(city_country):

  terms = [

    # Español
    "violencia política",
    "protestas violentas",
    "disturbios políticos",
    "enfrentamientos con la policía",
    "enfrentamientos con fuerzas de seguridad",
    "choques con la policía",
    "represión policial",
    "violencia en protestas",
    "manifestación violenta",
    "ataque político",
    "ataque extremista",
    "ataque terrorista",
    "violencia electoral",
    "fraude electoral protestas",
    "crisis política disturbios",
    "enfrentamientos entre manifestantes",
    "saqueos durante protestas",
    "protesta masiva disturbios",
    "conflicto político violento",

    # Portugués
    "violência política",
    "protestos violentos",
    "distúrbios políticos",
    "confrontos com a polícia",
    "confrontos com forças de segurança",
    "choques com a polícia",
    "repressão policial",
    "violência em protestos",
    "manifestação violenta",
    "ataque político",
    "ataque extremista",
    "ataque terrorista",
    "violência eleitoral",
    "protestos por fraude eleitoral",
    "crise política distúrbios",
    "confrontos entre manifestantes",
    "saques durante protestos",
    "protesto massivo distúrbios",
    "conflito político violento",

    # Inglés

    "political violence",
    "violent protest",
    "political unrest",
    "clashes with police",
    "political clashes",
    "mob attack politics",
    "political riots",
    "protest violence",
    "extremist attack politics"
]

  total = 0

  for term in terms:

      query = f"{term} {city_country}".replace(" ", "+")
      url = f"https://news.google.com/rss/search?q={query}"

      r = requests.get(url)
      soup = BeautifulSoup(r.text, "xml")

      items = soup.find_all("item")

      total += len(items)

  return total

In [ ]:
violence_news_index("Buenos Aires Argentina after:2023-10-10 before:2024-08-22")

460

# Latinobarómetro

In [ ]:
from pandas.io.stata import StataReader

with StataReader("https://github.com/LCaravaggio/Happiness_Polarization/raw/refs/heads/main/Latinobarometro_2024_Stata_esp_v20250817.dta") as reader:
    df = reader.read(convert_categoricals=False)
    labels = reader.value_labels()

In [ ]:
import re
from tqdm.auto import tqdm
import signal
import time


def clean_city_label(x):

    x = str(x)

    # eliminar prefijo país
    x = re.sub(r"^[A-Z]{2}:\s*", "", x)

    # reemplazos
    x = x.replace("-", " ")
    x = x.replace("/", " ")

    # eliminar cualquier [%...%] si aparece
    x = re.sub(r"\[%.*?%\]", "", x)

    # normalizar espacios
    x = re.sub(r"\s+", " ", x).strip()

    return x

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

signal.signal(signal.SIGALRM, timeout_handler)

results = []

pairs = df[["IDENPA","CIUDAD"]].drop_duplicates()

for _, row in tqdm(pairs.iloc[-1:].iterrows(), total=len(pairs)):

    country = labels["IDENPA"].get(row.IDENPA, str(row.IDENPA))
    city = labels["CIUDAD"].get(row.CIUDAD, str(row.CIUDAD))

    city = clean_city_label(city)
    country = clean_city_label(country)

    query = f"{city} {country} after:2023-10-10 before:2024-08-22"

    try:
        signal.alarm(120)   # máximo 120 segundos
        index = violence_news_index(query)
        signal.alarm(0)     # cancelar alarma
    except TimeoutException:
        index = None
    except:
        index = None

    results.append({
        "country": country,
        "city": city,
        "query": query,
        "violence_index": index
    })

    #print(country + "-" + city + ": " + str(index))
    #time.sleep(2)

  0%|          | 0/1062 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
from google.colab import files

violence_df = pd.DataFrame(results)
violence_df.to_csv("violence_news_index.csv", index=False)

files.download("violence_news_index.csv")

# Leer

In [1]:
import pandas as pd

violence=pd.read_csv('https://raw.githubusercontent.com/LCaravaggio/Happiness_Polarization/refs/heads/main/Political%20Violence%20Index.csv')

In [2]:
!wget https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
!unzip latinobarometro-2024-stata-v20250817.zip

--2026-06-01 19:44:42--  https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
Resolving www.latinobarometro.org (www.latinobarometro.org)... 80.28.53.190
Connecting to www.latinobarometro.org (www.latinobarometro.org)|80.28.53.190|:443... connected.
HTTP request sent, awaiting response... 200 200
Length: 6691592 (6.4M) [application/x-zip-compressed]
Saving to: ‘latinobarometro-2024-stata-v20250817.zip’

latinobarometro-202 100%[===================>]   6.38M  6.52MB/s    in 1.0s    

2026-06-01 19:44:44 (6.52 MB/s) - ‘latinobarometro-2024-stata-v20250817.zip’ saved [6691592/6691592]

Archive:  latinobarometro-2024-stata-v20250817.zip
  inflating: Latinobarometro_2024_Stata_eng_v20250817.dta  
  inflating: Latinobarometro_2024_Stata_esp_v20250817.dta  
  inflating: Latinobarometro_2024_Cuestionario_esp.pdf  
  inflating: Latinobarometro_2024_Cuestionario_eng.pdf  


In [3]:
import pandas as pd

# abrir reader
reader = pd.read_stata(
    "Latinobarometro_2024_Stata_esp_v20250817.dta",
    convert_categoricals=False,
    iterator=True
)

# dataframe
df24 = reader.read()

# labels de variables
variable_labels = reader.variable_labels()

# TODOS los value labels
value_labels = reader.value_labels()

In [4]:
import numpy as np

conversion = {

    4: 1,  # para nada
    3: 2,  # no muy
    2: 3,  # bastante
    1: 4   # muy
}


def convertir(val):

    try:

        num = float(val)

        # missings típicos latinobarómetro
        if num in [97, 98, 99]:
            return np.nan

        return num

    except:

        return np.nan

df24["life_satisfaction"] = (
    df24["P1ST"]
    .map(conversion)
)

df24['engagement'] = (
    df24["P36STGBS"].map(conversion)
)

df24["pol_scale"] = (
    df24["P16ST"]
    .apply(convertir)
)

# distancia al centro
df24["polarization"] = (
    np.abs(df24["pol_scale"] - 5)
)

/tmp/ipykernel_9578/3264565117.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df24["life_satisfaction"] = (
/tmp/ipykernel_9578/3264565117.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df24['engagement'] = (
/tmp/ipykernel_9578/3264565117.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

In [12]:
import pandas as pd
from scipy.stats import pearsonr

# =========================================================
# COLAPSAR LIFE SATISFACTION A NIVEL CIUDAD
# =========================================================

city_life = (
    df24
    .groupby("CIUDAD", as_index=False)[["life_satisfaction", "engagement", "polarization"]]
    .mean()
)

ciudad_labels = value_labels["CIUDAD"]

city_life["city"] = city_life["CIUDAD"].map(ciudad_labels)
violence['city']=violence['city_name']



In [13]:
import pandas as pd
import unicodedata
import re

def clean_city(x):

    if pd.isna(x):
        return None

    x = str(x).lower()

    # quitar prefijo tipo "AR:"
    x = re.sub(r"^[a-z]{2}:\s*", "", x)

    # quitar tildes
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )

    # reemplazar separadores por espacio
    x = re.sub(r"[-_/]", " ", x)

    # sacar puntuación
    x = re.sub(r"[^a-z0-9\s]", "", x)

    # espacios múltiples
    x = re.sub(r"\s+", " ", x)

    return x.strip()

In [15]:
city_life["city_clean"] = city_life["city"].apply(clean_city)

violence["city_clean"] = violence["city_name"].apply(clean_city)

In [21]:
# =========================================================
# MERGE
# =========================================================


df_corr = city_life.merge(
    violence,
    on="city_clean",
    how="inner"
)

from scipy.stats import pearsonr
import pandas as pd

variables = [
    "life_satisfaction",
    "engagement",
    "polarization"
]

results = []

for var in variables:

    tmp = df_corr[
        [var, "news_index_3"]
    ].dropna()

    if len(tmp) < 6:
        continue

    r, p = pearsonr(
        tmp["news_index_3"],
        tmp[var]
    )

    results.append({
        "variable": var,
        "pearson_r": r,
        "p_value": p,
        "n": len(tmp)
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("pearson_r", ascending=False)
)

print('Comparativa con News Index:')
results_df

Comparativa con News Index:


,variable,pearson_r,p_value,n
1,engagement,0.074889,0.014551,1064
2,polarization,-0.009256,0.763201,1062
0,life_satisfaction,-0.079121,0.009827,1064
